# GPU Experiment 3: Placebo Control (v_rand) & RAG Baseline Evaluation
This notebook evaluates:
1. **Random-Direction Placebo Control ($v_{\text{rand}}$)** to prove causal directionality.
2. **Drug Formulary RAG Baseline** to measure reference preference, latency, and context length trade-offs.

In [1]:
!pip install -q -U transformers accelerate bitsandbytes evaluate bert_score rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 96.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 25.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.0/41.0 MB 50.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 29.8 MB/s eta 0:00:00


In [2]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import json
import numpy as np
import glob
import time
from evaluate import load
from tqdm import tqdm
from rank_bm25 import BM25Okapi

model_id = "Qwen/Qwen2.5-7B-Instruct"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    device_map="auto",
    quantization_config=BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)
)
bertscore = load("bertscore")

config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

In [3]:
# Load Random Vector (v_rand.pt)
v_rand_files = glob.glob('/kaggle/input/**/v_rand.pt', recursive=True)
if not v_rand_files:
    v_rand_files = glob.glob('/kaggle/input/**/*.pt', recursive=True)

if not v_rand_files:
    raise FileNotFoundError("Could not find v_rand.pt in /kaggle/input!")

print(f"Found v_rand file at: {v_rand_files[0]}")
v_rand_data = torch.load(v_rand_files[0], map_location='cpu')
if isinstance(v_rand_data, dict):
    v_rand = v_rand_data.get('steering_vector', v_rand_data.get('v_rand', list(v_rand_data.values())[0]))
else:
    v_rand = v_rand_data
v_rand = v_rand.to(model.device).to(torch.float16)
# Ensure unit norm
v_rand = v_rand / torch.norm(v_rand)
print("Loaded v_rand vector successfully!")

# Load Test Data
json_files = glob.glob('/kaggle/input/**/vietnamese_medical_halueval_15k_specialized.json', recursive=True)
if not json_files:
    json_files = glob.glob('/kaggle/input/**/*.json', recursive=True)

with open(json_files[0], 'r', encoding='utf-8') as f:
    data = json.load(f)
print(f"Loaded dataset from: {json_files[0]}")
test_data = data[-500:]

def format_prompt_no_ctx(q):
    messages = [{"role": "system", "content": "You are a helpful and accurate medical assistant."}, {"role": "user", "content": q}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def format_prompt_rag(q, context):
    sys_prompt = f"You are a helpful medical assistant. Use the following context to answer accurately:\nContext: {context}"
    messages = [{"role": "system", "content": sys_prompt}, {"role": "user", "content": q}]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

Found v_rand file at: /kaggle/input/datasets/anhemgithom/steering-phase1-artifacts/v_rand.pt
Loaded v_rand vector successfully!
Loaded dataset from: /kaggle/input/datasets/anhemgithom/vnese-data/vietnamese_medical_halueval_15k_specialized.json


In [4]:
print("🚀 Running Placebo Control (v_rand steering)...")

step_counter = [0]
def steering_hook(module, input, output):
    step_counter[0] += 1
    t = step_counter[0]
    if t <= 16:
        alpha_t = 18.0 * (1.0 - (t - 1) / 16.0)
        if isinstance(output, tuple):
            h = output[0]
            v = v_rand.to(device=h.device, dtype=h.dtype)
            h[:, -1, :] += alpha_t * v
            return (h,) + output[1:]
        else:
            v = v_rand.to(device=output.device, dtype=output.dtype)
            output[:, -1, :] += alpha_t * v
            return output
    return output

hook_handle = model.model.layers[8].register_forward_hook(steering_hook)

placebo_gen, refs, hals = [], [], []
for item in tqdm(test_data, desc="Placebo Evaluation"):
    step_counter[0] = 0
    prompt = format_prompt_no_ctx(item['question'])
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        
    gen_text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    placebo_gen.append(gen_text)
    refs.append(item.get('right_answer', item.get('positive_answer')))
    hals.append(item['hallucinated_answer'])

hook_handle.remove()

bs_ref = bertscore.compute(predictions=placebo_gen, references=refs, model_type="bert-base-multilingual-cased")['f1']
bs_hal = bertscore.compute(predictions=placebo_gen, references=hals, model_type="bert-base-multilingual-cased")['f1']
placebo_acc = sum(1 for r, h in zip(bs_ref, bs_hal) if r > h) / len(test_data) * 100
placebo_bs = np.mean(bs_ref)

print(f"Placebo (v_rand): Accuracy = {placebo_acc:.2f}%, BERTScore = {placebo_bs:.4f}")

🚀 Running Placebo Control (v_rand steering)...



Placebo Evaluation: 100%|██████████| 500/500 [1:45:37<00:00, 12.68s/it]


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Placebo (v_rand): Accuracy = 62.60%, BERTScore = 0.6945


In [5]:
print("🚀 Running RAG Baseline Evaluation...")

rag_gen, rag_refs, rag_hals, latencies, prompt_lens = [], [], [], [], []

for item in tqdm(test_data, desc="RAG Evaluation"):
    context = item.get('knowledge_context', item.get('context', ''))
    prompt = format_prompt_rag(item['question'], context)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    prompt_len = inputs.input_ids.shape[1]
    prompt_lens.append(prompt_len)
    
    t0 = time.time()
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=200, do_sample=False, pad_token_id=tokenizer.pad_token_id)
    latencies.append(time.time() - t0)
    
    gen_text = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True)
    rag_gen.append(gen_text)
    rag_refs.append(item.get('right_answer', item.get('positive_answer')))
    rag_hals.append(item['hallucinated_answer'])

rag_bs_ref = bertscore.compute(predictions=rag_gen, references=rag_refs, model_type="bert-base-multilingual-cased")['f1']
rag_bs_hal = bertscore.compute(predictions=rag_gen, references=rag_hals, model_type="bert-base-multilingual-cased")['f1']
rag_acc = sum(1 for r, h in zip(rag_bs_ref, rag_bs_hal) if r > h) / len(test_data) * 100
rag_bs = np.mean(rag_bs_ref)
mean_latency = np.mean(latencies)
mean_prompt_len = np.mean(prompt_lens)

print(f"RAG Baseline: Accuracy = {rag_acc:.2f}%, BERTScore = {rag_bs:.4f}")
print(f"RAG Mean Latency = {mean_latency:.2f}s, Mean Prompt Length = {mean_prompt_len:.1f} tokens")

results_summary = {
    "placebo_v_rand": {"accuracy": placebo_acc, "bertscore": placebo_bs},
    "rag_baseline": {"accuracy": rag_acc, "bertscore": rag_bs, "mean_latency": mean_latency, "mean_prompt_len": mean_prompt_len}
}

with open("placebo_and_rag_results.json", "w") as f:
    json.dump(results_summary, f, indent=2)
print("Saved placebo_and_rag_results.json successfully!")

🚀 Running RAG Baseline Evaluation...



RAG Evaluation: 100%|██████████| 500/500 [1:50:39<00:00, 13.28s/it]


RAG Baseline: Accuracy = 79.20%, BERTScore = 0.7382
RAG Mean Latency = 13.27s, Mean Prompt Length = 304.4 tokens
Saved placebo_and_rag_results.json successfully!
